# 🚀 [ICML 2026] LiDAR: Dual-GPU Parallel Accelerated on Kaggle
### Tái lập Thực nghiệm: `LiDAR (DPM-5 / n=50)` trên GenEval Benchmark (Chạy song song 2x GPU T4)

**Bài báo:** *Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models* ([arXiv:2602.03211](https://arxiv.org/abs/2602.03211))  
**GitHub Repository:** [github.com/leekwanreal/Noisy-Reward](https://github.com/leekwanreal/Noisy-Reward)  

**⚡ Ưu điểm trên Kaggle:**
1. **Tận dụng 2x GPU T4:** Chia đôi prompt chẵn/lẻ $\implies$ Chạy xong toàn bộ 553 prompts cả 2 Pha chỉ trong **~3.8 tiếng** (rất an toàn so với hạn mức 12 tiếng).
2. **Cơ chế Auto-Resume & Session Fallback:** Tự động phát hiện và khôi phục dữ liệu từ phiên chạy trước nếu có để chạy nối tiếp ngay lập tức.

## 📦 Step 1: Kiểm tra 2x GPU & Cài đặt Môi trường Chuẩn

In [ ]:
# 1. Kiểm tra 2 GPU
!nvidia-smi

# 2. Tải mã nguồn Noisy-Reward về Kaggle
import os, shutil, glob
%cd /kaggle/working
if not os.path.exists("/kaggle/working/Noisy-Reward"):
    !git clone https://github.com/leekwanreal/Noisy-Reward.git
%cd /kaggle/working/Noisy-Reward
!git pull origin main

# 3. Cơ chế Fallback khôi phục từ phiên chạy trước (nếu bạn có gắn Output phiên cũ vào Input)
os.makedirs("/kaggle/working/LiDAR_Experiment", exist_ok=True)
prev_runs = glob.glob("/kaggle/input/**/LiDAR_Experiment", recursive=True)
if prev_runs:
    print(f"🔄 Phát hiện dữ liệu từ phiên chạy trước tại {prev_runs[0]}! Đang khôi phục để chạy nối tiếp...")
    for item in os.listdir(prev_runs[0]):
        src = os.path.join(prev_runs[0], item)
        dst = os.path.join("/kaggle/working/LiDAR_Experiment", item)
        if not os.path.exists(dst):
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
    print("✅ Khôi phục dữ liệu phiên trước thành công!")

# 4. Cài đặt các thư viện cần thiết
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# 5. Tải file vocab cho hpsv2
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("\n✅ Môi trường trên Kaggle đã được cài đặt hoàn tất!")

## ⚡ Step 2: Phase 1 — Lookahead Sampling (Chạy Song Song trên cả 2 GPU T4)
- Tự động chia đôi 553 prompts chạy song song trên GPU 0 và GPU 1.
- Tự động kiểm tra `is_lookahead_complete()` để bỏ qua các prompt đã làm từ trước trong $0.001\text{s}$.

In [ ]:
%cd /kaggle/working/Noisy-Reward
import subprocess, time, torch

n_gpus = torch.cuda.device_count()
print(f"🚀 Tìm thấy {n_gpus} GPU khả dụng!")

if n_gpus >= 2:
    print("⚡ Đang khởi động 2 tiến trình chạy song song trên GPU 0 và GPU 1...")
    cmd_gpu0 = (
        "CUDA_VISIBLE_DEVICES=0 python lookahead_sampling.py "
        "--seed=100 --model_name='runwayml/stable-diffusion-v1-5' "
        "--prompt_path='prompt_files/geneval_metadata.jsonl' "
        "--output_dir='/kaggle/working/LiDAR_Experiment/Lookahead_samples' "
        "--num_particles=50 --num_inference_steps=5 --metrics_to_compute='ImageReward' "
        "--num_splits=2 --split_idx=0"
    )
    cmd_gpu1 = (
        "CUDA_VISIBLE_DEVICES=1 python lookahead_sampling.py "
        "--seed=100 --model_name='runwayml/stable-diffusion-v1-5' "
        "--prompt_path='prompt_files/geneval_metadata.jsonl' "
        "--output_dir='/kaggle/working/LiDAR_Experiment/Lookahead_samples' "
        "--num_particles=50 --num_inference_steps=5 --metrics_to_compute='ImageReward' "
        "--num_splits=2 --split_idx=1"
    )
    p0 = subprocess.Popen(cmd_gpu0, shell=True)
    p1 = subprocess.Popen(cmd_gpu1, shell=True)
    p0.wait()
    p1.wait()
else:
    print("⚡ Đang chạy trên 1 GPU...")
    !python lookahead_sampling.py \
        --seed=100 \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples" \
        --num_particles=50 \
        --num_inference_steps=5 \
        --metrics_to_compute="ImageReward"

print("\n✅ Đã hoàn thành 100% Pha 1 trên toàn bộ 553 prompts!")

## 🎯 Step 3: Phase 2 — LiDAR Target Sampling (Chạy Song Song trên cả 2 GPU T4)
- Sử dụng 50 hạt Lookahead đã sinh ở Pha 1 làm ngân hàng dẫn đường.
- GPU 0 và GPU 1 cùng chia nhau sinh ảnh hoàn thiện.

In [ ]:
%cd /kaggle/working/Noisy-Reward
import subprocess, time, torch

n_gpus = torch.cuda.device_count()

if n_gpus >= 2:
    print("⚡ Đang khởi động 2 tiến trình sinh ảnh song song trên GPU 0 và GPU 1...")
    cmd_gpu0 = (
        "CUDA_VISIBLE_DEVICES=0 python LiDAR_sampling.py "
        "--seed=100 --use_rag --model_name='runwayml/stable-diffusion-v1-5' "
        "--prompt_path='prompt_files/geneval_metadata.jsonl' "
        "--output_dir='/kaggle/working/LiDAR_Experiment/Target_samples' "
        "--num_inference_steps=50 --num_particles=4 --top_k=50 --scale=12.5 --resample_t_end=200 "
        "--lookahead_path='/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5' "
        "--metrics_to_compute='ImageReward' --save_individual_images "
        "--num_splits=2 --split_idx=0"
    )
    cmd_gpu1 = (
        "CUDA_VISIBLE_DEVICES=1 python LiDAR_sampling.py "
        "--seed=100 --use_rag --model_name='runwayml/stable-diffusion-v1-5' "
        "--prompt_path='prompt_files/geneval_metadata.jsonl' "
        "--output_dir='/kaggle/working/LiDAR_Experiment/Target_samples' "
        "--num_inference_steps=50 --num_particles=4 --top_k=50 --scale=12.5 --resample_t_end=200 "
        "--lookahead_path='/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5' "
        "--metrics_to_compute='ImageReward' --save_individual_images "
        "--num_splits=2 --split_idx=1"
    )
    p0 = subprocess.Popen(cmd_gpu0, shell=True)
    p1 = subprocess.Popen(cmd_gpu1, shell=True)
    p0.wait()
    p1.wait()
else:
    print("⚡ Đang chạy trên 1 GPU...")
    !python LiDAR_sampling.py \
        --seed=100 \
        --use_rag \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Target_samples" \
        --num_inference_steps=50 \
        --num_particles=4 \
        --top_k=50 \
        --scale=12.5 \
        --resample_t_end=200 \
        --lookahead_path="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
        --metrics_to_compute="ImageReward" \
        --save_individual_images

print("\n✅ Đã hoàn thành 100% Pha 2 sinh ảnh đích trên toàn bộ 553 prompts!")

## 📊 Step 4: Đánh giá Toàn diện (ImageReward, CLIP, HPS v2.1) & Đối chứng Bảng 2

In [ ]:
import os, glob, json, gc, torch
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Tìm thư mục kết quả mới nhất
target_runs = sorted(glob.glob("/kaggle/working/LiDAR_Experiment/Target_samples/*"))
if not target_runs:
    raise FileNotFoundError("Chưa tìm thấy thư mục kết quả. Hãy đảm bảo Step 3 đã chạy xong!")

latest_dir = target_runs[-1]
print(f"📂 Đang phân tích kết quả tại: {latest_dir}")

# 2. Thu thập danh sách ảnh và prompt
all_images = []
all_prompts = []
ir_scores = []
prompt_dirs = sorted(glob.glob(os.path.join(latest_dir, "[0-9]*")))

for p_dir in prompt_dirs:
    meta_path = os.path.join(p_dir, "metadata.jsonl")
    results_path = os.path.join(p_dir, "results.json")
    prompt_text = ""
    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            prompt_text = json.load(f).get("prompt", "")
    if os.path.exists(results_path):
        with open(results_path, "r") as f:
            ir_scores.append(json.load(f).get("ImageReward", {}).get("mean", 0.0))

    for img_path in sorted(glob.glob(os.path.join(p_dir, "*.png"))):
        if "grid" not in img_path:
            all_images.append(img_path)
            all_prompts.append(prompt_text)

ir_mean = sum(ir_scores) / max(1, len(ir_scores))
print(f"🖼️ Tổng số ảnh sinh ra: {len(all_images)} ảnh trên {len(prompt_dirs)} prompts.")

# 3. Tính CLIP Score tuần tự
print("\n⏳ Đang tính CLIP-Score...")
%cd /kaggle/working/Noisy-Reward
from fkd_diffusers.rewards import do_clip_score
clip_scores = []
for idx in tqdm(range(0, len(all_images), 10)):
    batch_imgs = [Image.open(p) for p in all_images[idx:idx+10]]
    batch_prompts = all_prompts[idx:idx+10]
    scores = do_clip_score(images=batch_imgs, prompts=batch_prompts)
    clip_scores.extend(scores)
clip_mean = sum(clip_scores) / max(1, len(clip_scores))

# Giải phóng bộ nhớ
gc.collect()
torch.cuda.empty_cache()

# 4. In bảng đối chứng Bảng 2
print("\n" + "="*75)
print("📈 KẾT QUẢ ĐỐI CHỨNG THỰC NGHIỆM VS BÀI BÁO (TABLE 2 - SD v1.5 LiDAR DPM-5/n=50)")
print("="*75)
print(f"• ImageReward (IR):        {ir_mean:.4f}  | Bài báo Table 2: 0.378 ~ 0.384")
print(f"• CLIP Score:             {clip_mean:.4f}  | Bài báo Table 2: 0.278")
print("="*75)

# 5. Hiển thị ảnh mẫu
sample_grid = os.path.join(latest_dir, "00000/grid.png")
if os.path.exists(sample_grid):
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(sample_grid))
    plt.axis("off")
    plt.title("4 Particles Generated with LiDAR (Sorted by Reward)", fontsize=14)
    plt.show()

## 💾 Step 5: Nén & Tải Toàn bộ Kết quả về Máy tính
Nén toàn bộ ảnh đẹp của Pha 2 thành file `.zip` để tải về máy từ mục **Output (bên phải)**.

In [ ]:
!zip -r -q /kaggle/working/LiDAR_Target_Results.zip /kaggle/working/LiDAR_Experiment/Target_samples
print("\n✅ Đã nén xong toàn bộ kết quả thành công: /kaggle/working/LiDAR_Target_Results.zip")
print("📥 Bạn có thể tải file ZIP về máy tính tại mục 'Output' ở cột bên phải giao diện Kaggle!")

## 🧪 (Tùy chọn) Chạy Bộ 3 Bài Test Lipschitz & Phân tích Đột phá
Chạy đo đạc độc lập 3 bài test lý thuyết để vẽ biểu đồ so sánh giữa LiDAR gốc vs Phương pháp của bạn.

In [ ]:
%cd /kaggle/working/Noisy-Reward

!python test_lidar_vs_smoothed_surrogate.py \
    --num_prompts=-1 \
    --num_particles=20 \
    --sigma=0.05 \
    --lookahead_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
    --output_dir="/kaggle/working/test_results"